In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from sklearn.ensemble import IsolationForest

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
from IPython.display import display, Markdown

# Semana 10B — Detección basada en Descomposición (Laboratorio)

**Objetivo:** Aplicar STL e Isolation Forest al dataset ClimaLab
y consolidar un DataFrame de flags de anomalías para usar en el
bloque de imputación.

**Ejercicios:**

1. STL sobre `tdb` horaria
2. STL sobre `ghi` horaria
3. Comparación de umbrales en residuales
4. Isolation Forest multivariado
5. DataFrame de flags consolidado

In [ ]:
df = pd.read_parquet("data/ClimaLab_2023-05-31_2025-06-20.parquet")

In [ ]:
tdb_horaria = df["tdb"].resample("1h").mean().dropna()
ghi_horaria = df["ghi"].resample("1h").mean().dropna()

---
## 1. STL sobre `tdb` horaria (period=24)

Descomponemos la temperatura horaria con STL.  El período = 24
captura el ciclo diurno.  Con `robust=True`, los outliers no
contaminan la estimación de tendencia ni estacionalidad.

In [ ]:
stl_tdb = STL(tdb_horaria, period=24, robust=True).fit()

fig1 = stl_tdb.plot()
fig1.set_size_inches(13, 10)
fig1.suptitle("STL — tdb horaria (period=24, robust=True)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
resid_tdb = stl_tdb.resid.dropna()

fig1b, (ax1a, ax1b) = plt.subplots(1, 2, figsize=(12, 4))

ax1a.hist(resid_tdb.values, bins=100, density=True, color="steelblue",
          alpha=0.7, edgecolor="white")
ax1a.set_xlabel("Residuo (°C)")
ax1a.set_ylabel("Densidad")
ax1a.set_title(f"Distribución de residuales (σ = {np.std(resid_tdb.values):.2f})")

# QQ-plot rápido
from scipy import stats as scipy_stats
osm, osr = scipy_stats.probplot(resid_tdb.values, dist="norm")
ax1b.scatter(osm[0], osm[1], c="steelblue", s=3, alpha=0.3)
slope, intercept = np.polyfit(osm[0], osm[1], 1)
x_line = np.array([osm[0].min(), osm[0].max()])
ax1b.plot(x_line, slope * x_line + intercept, "k--", lw=1.5)
ax1b.set_xlabel("Cuantiles teóricos")
ax1b.set_ylabel("Cuantiles observados")
ax1b.set_title("QQ-plot de residuales")

plt.tight_layout()
fig1b
from IPython.display import display, Markdown

> **Observa:** Los residuales son aproximadamente normales pero con
> colas pesadas (picos en el QQ-plot).  Esas colas son los candidatos
> a outlier.

---
## 2. STL sobre `ghi` horaria

La irradiancia tiene estacionalidad fuerte (día/noche) y muchos NaN
(noches).  Filtramos las horas con datos y aplicamos STL.
El residuo puede revelar anomalías del sensor no evidentes
en la serie bruta.

In [ ]:
# Filtrar NaN y valores muy cercanos a 0 (noches)
ghi_filtrada = ghi_horaria[ghi_horaria > 1].dropna()

stl_ghi = STL(ghi_filtrada, period=24, robust=True).fit()

fig2 = stl_ghi.plot()
fig2.set_size_inches(13, 10)
fig2.suptitle("STL — ghi horaria (solo horas diurnas, period=24)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

> El residuo de GHI puede mostrar:
> - Lecturas anormalmente bajas a mediodía (sensor sucio, sombra)
> - Picos que exceden la irradiancia extraterrestre (sensor defectuoso)
> - Estas anomalías no son evidentes en la serie bruta donde la
>   estacionalidad domina.

---
## 3. Comparación de umbrales sobre residuales de `tdb`

Sobre los residuales de STL, comparamos tres umbrales:
- **(a) ±3σ global:** asume normalidad y varianza constante
- **(b) Percentiles 0.5% / 99.5%:** no asume normalidad
- **(c) MAD con factor 3:** robusto a colas pesadas

In [ ]:
rv = resid_tdb.values

# (a) ±3σ
sigma_r = np.std(rv)
mu_r = np.mean(rv)
umbral_3s_lo = mu_r - 3 * sigma_r
umbral_3s_hi = mu_r + 3 * sigma_r
out_3s = (rv < umbral_3s_lo) | (rv > umbral_3s_hi)

# (b) Percentiles
p_lo = np.percentile(rv, 0.5)
p_hi = np.percentile(rv, 99.5)
out_pct = (rv < p_lo) | (rv > p_hi)

# (c) MAD
med_r = np.median(rv)
mad_r = np.median(np.abs(rv - med_r))
mad_scaled = 1.4826 * mad_r
umbral_mad_lo = med_r - 3 * mad_scaled
umbral_mad_hi = med_r + 3 * mad_scaled
out_mad = (rv < umbral_mad_lo) | (rv > umbral_mad_hi)

display(Markdown(f"""
### Conteos de detección

| Umbral | Límite inferior | Límite superior | # Detecciones |
|:---|---:|---:|---:|
| ±3σ | {umbral_3s_lo:.2f} | {umbral_3s_hi:.2f} | {out_3s.sum():,} |
| P0.5/P99.5 | {p_lo:.2f} | {p_hi:.2f} | {out_pct.sum():,} |
| MAD ×3 | {umbral_mad_lo:.2f} | {umbral_mad_hi:.2f} | {out_mad.sum():,} |
"""))

In [ ]:
fig3, ax3 = plt.subplots(figsize=(13, 5))

step = max(1, len(resid_tdb) // 30000)
idx = resid_tdb.index[::step]
vals = resid_tdb.values[::step]

ax3.scatter(idx, vals, c="gray", s=1, alpha=0.2)

# Umbrales como líneas
ax3.axhline(mu_r + 3 * sigma_r, color="crimson", ls="--", lw=1.5, label="±3σ")
ax3.axhline(mu_r - 3 * sigma_r, color="crimson", ls="--", lw=1.5)
ax3.axhline(p_hi, color="darkorange", ls=":", lw=1.5, label="P0.5/P99.5")
ax3.axhline(p_lo, color="darkorange", ls=":", lw=1.5)
ax3.axhline(umbral_mad_hi, color="seagreen", ls="-.", lw=1.5, label="MAD ×3")
ax3.axhline(umbral_mad_lo, color="seagreen", ls="-.", lw=1.5)

ax3.set_ylabel("Residuo (°C)")
ax3.set_xlabel("Fecha")
ax3.set_title("Residuales de STL con tres umbrales superpuestos")
ax3.legend()
plt.tight_layout()
plt.show()

> **Discusión:**
> - **MAD** suele dar el umbral más estrecho → más detecciones (más agresivo).
> - **±3σ** puede ser más laxo si las colas pesadas inflan σ.
> - **Percentiles** dan un número fijo de detecciones (~1% del total).
> - No hay umbral "correcto" — la elección depende de las consecuencias
>   de falsos positivos vs falsos negativos en el contexto de aplicación.

---
## 4. Isolation Forest multivariado

Aplicamos Isolation Forest sobre 4 variables simultáneamente
(`tdb`, `rh`, `ws`, `p_atm` horarios).  Esto puede captar
anomalías que son normales en cada variable individual pero
cuya **combinación** es inusual.

In [ ]:
# Preparar datos multivariados horarios
df_horario = df[["tdb", "rh", "ws", "p_atm"]].resample("1h").mean().dropna()

iforest = IsolationForest(contamination=0.01, random_state=42, n_jobs=-1)
pred_if = iforest.fit_predict(df_horario.values)
scores_if = iforest.decision_function(df_horario.values)

out_iforest = pred_if == -1
n_if = out_iforest.sum()

display(Markdown(f"""
**Isolation Forest** (contamination=1%, 4 variables):
**{n_if:,}** anomalías detectadas de {len(df_horario):,} registros horarios.
"""))

In [ ]:
fig4, axes4 = plt.subplots(2, 2, figsize=(13, 8))

variables = ["tdb", "rh", "ws", "p_atm"]
colores_normal = ["steelblue", "steelblue", "steelblue", "steelblue"]

step = max(1, len(df_horario) // 30000)

for ax, var in zip(axes4.flat, variables):
    vals = df_horario[var].values
    idx = df_horario.index

    ax.scatter(idx[::step][~out_iforest[::step]],
               vals[::step][~out_iforest[::step]],
               c="steelblue", s=1, alpha=0.15)
    ax.scatter(idx[::step][out_iforest[::step]],
               vals[::step][out_iforest[::step]],
               c="crimson", s=5, alpha=0.5, zorder=5)
    ax.set_ylabel(var)
    ax.set_title(var)

plt.suptitle(
    "Isolation Forest — anomalías multivariadas (rojo)",
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()

> **Observa:** Algunas anomalías marcadas en rojo pueden parecer
> normales al mirar una sola variable, pero su **combinación** es
> inusual (e.g., temperatura alta + humedad alta + viento cero
> → posible fallo del abrigo del sensor).
>
> Isolation Forest complementa los métodos univariados (STL, z-score)
> al captar este tipo de anomalías multivariadas.

---
## 5. DataFrame de flags consolidado

Combinamos todos los métodos de detección (semanas 9 y 10) en un
solo DataFrame con columnas booleanas.  Este artefacto se usará
en el **bloque 5** para decidir qué datos reemplazar con NaN
antes de imputar.

In [ ]:
# Resamplear todo a horario para tener un índice común
tdb_h = df["tdb"].resample("1h").mean()
rh_h = df["rh"].resample("1h").mean()
patm_h = df["p_atm"].resample("1h").mean()

# 1. Reglas físicas
flag_fisico = (rh_h < 0) | (patm_h < 500) | (patm_h > 1100) | (tdb_h > 50) | (tdb_h < -10)
flag_fisico = flag_fisico.reindex(df_horario.index).fillna(False)

# 2. Z-score estacional sobre tdb horaria
grupos = tdb_h.dropna().groupby([tdb_h.dropna().index.hour, tdb_h.dropna().index.month])
z_estac = ((tdb_h.dropna() - grupos.transform("mean")) / grupos.transform("std")).fillna(0)
flag_z_estac = (np.abs(z_estac) > 3).reindex(df_horario.index).fillna(False)

# 3. MAD sobre tdb horaria
med_tdb = tdb_h.median()
mad_tdb = np.median(np.abs(tdb_h.dropna() - med_tdb))
mad_s = 1.4826 * mad_tdb
flag_mad = (np.abs(tdb_h - med_tdb) / mad_s > 3).reindex(df_horario.index).fillna(False)

# 4. STL (residuales)
resid = stl_tdb.resid
med_r = np.median(resid.dropna())
mad_r = np.median(np.abs(resid.dropna() - med_r))
flag_stl = (np.abs(resid - med_r) / (1.4826 * mad_r) > 3).reindex(df_horario.index).fillna(False)

# 5. Isolation Forest
flag_iforest = pd.Series(out_iforest, index=df_horario.index)

flags = pd.DataFrame({
    "outlier_fisico": flag_fisico,
    "outlier_zscore_estacional": flag_z_estac,
    "outlier_mad": flag_mad,
    "outlier_stl": flag_stl,
    "outlier_iforest": flag_iforest,
})
flags["alguno"] = flags.any(axis=1)

n_total = len(flags)
rows_md = ""
col_names = {
    "outlier_fisico": "Reglas físicas",
    "outlier_zscore_estacional": "Z-score estacional",
    "outlier_mad": "MAD",
    "outlier_stl": "STL + MAD",
    "outlier_iforest": "Isolation Forest",
    "alguno": "**Algún método**",
}
col_list = list(col_names.keys())
idx = 0
while idx < len(col_list):
    col = col_list[idx]
    label = col_names[col]
    count = flags[col].sum()
    pct = 100 * count / n_total
    rows_md += f"| {label} | {count:,} | {pct:.2f}% |\n"
    idx += 1
display(Markdown(f"""
### Resumen de flags (resolución horaria)

| Método | # Anomalías | % del total |
|:---|---:|---:|
{rows_md}

Total de registros horarios: **{n_total:,}**
"""))

> Este DataFrame `flags` se puede exportar y cargar en las sesiones
> de imputación (semanas 11–13) para reemplazar los registros
> marcados por NaN antes de aplicar los métodos de relleno.

---
## Resumen de la sesión

| Ejercicio | Hallazgo principal |
|:---|:---|
| **STL tdb** | Los residuales son ~normales con colas pesadas; los picos extremos son candidatos a outlier |
| **STL ghi** | Revela anomalías del sensor (lecturas nocturnas, caídas a mediodía) ocultas en la serie bruta |
| **Umbrales** | MAD es más agresivo; ±3σ puede ser laxo con colas pesadas; percentiles dan % fijo |
| **Isolation Forest** | Detecta anomalías multivariadas que métodos univariados pierden |
| **Flags** | DataFrame consolidado con 5 columnas booleanas, listo para el bloque de imputación |

**Próxima semana (11):** Imputación de datos faltantes —
forward fill, interpolación, día análogo.